In [90]:
#Importar librerías
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
import numpy as np

2. Cargar los datos y ver las columnas

In [91]:
EmpleadosAttrition = pd.read_csv("empleadosRETO.csv")
EmpleadosAttrition.columns

Index(['Age', 'BusinessTravel', 'Department', 'DistanceFromHome', 'Education',
       'EducationField', 'EmployeeCount', 'EmployeeNumber',
       'EnvironmentSatisfaction', 'Gender', 'JobInvolvement', 'JobLevel',
       'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome',
       'NumCompaniesWorked', 'HiringDate', 'Over18', 'OverTime',
       'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction',
       'StandardHours', 'TotalWorkingYears', 'TrainingTimesLastYear',
       'WorkLifeBalance', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
       'Attrition'],
      dtype='object')

3. Eliminar columnas sin relación

In [92]:
EmpleadosAttrition = EmpleadosAttrition.drop(columns = ['EmployeeCount','EmployeeNumber', 'Over18', 'StandardHours'])
EmpleadosAttrition

,Age,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,JobInvolvement,JobLevel,...,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsInCurrentRole,YearsSinceLastPromotion,Attrition
0,50,Travel_Rarely,Research & Development,1 km,2,Medical,4,Male,3,4,...,No,22,4,3,32,1,2,4,1,No
1,36,Travel_Rarely,Research & Development,6 km,2,Medical,2,Male,3,2,...,No,20,4,4,7,0,3,2,0,No
2,21,Travel_Rarely,Sales,7 km,1,Marketing,2,Male,3,1,...,No,13,3,2,1,3,3,0,1,Yes
3,52,Travel_Rarely,Research & Development,7 km,4,Life Sciences,2,Male,3,3,...,No,19,3,4,18,4,3,6,4,No
4,33,Travel_Rarely,Research & Development,15 km,1,Medical,2,Male,3,3,...,Yes,12,3,4,15,2,4,6,7,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,33,Travel_Rarely,Research & Development,14 km,3,Medical,3,Male,3,1,...,Yes,13,3,3,8,2,1,4,0,Yes
396,31,Travel_Rarely,Sales,20 km,3,Life Sciences,2,Female,1,2,...,Yes,11,3,3,4,2,3,2,2,Yes
397,37,Travel_Frequently,Research & Development,11 km,3,Other,2,Male,3,3,...,Yes,14,3,3,10,1,3,8,0,No
398,38,Travel_Rarely,Research & Development,4 km,2,Medical,4,Female,3,1,...,No,19,3,4,7,5,2,0,0,No


4. Extraer el año

In [93]:
# convertir a entero la variable HiringDate
EmpleadosAttrition['HiringDate'] = EmpleadosAttrition['HiringDate'].str.strip()

In [94]:
# Convertir a años extrallendo el año de cada valor
EmpleadosAttrition['Year'] = pd.to_datetime(
    EmpleadosAttrition['HiringDate'],
    errors='coerce'
).dt.year

In [95]:
# Diferencia de años a partir de la nueva variable creada con respecto al 2018
EmpleadosAttrition['YearsAtCompany'] = 2018 - EmpleadosAttrition['Year']

5.	La DistanceFromHome está dada en kilómetros, pero tiene las letras “km” al final y así no puede ser entera:

In [96]:
# Renombrar
EmpleadosAttrition = EmpleadosAttrition.rename(
    columns={'DistanceFromHome': 'DistanceFromHome_km'}
)

In [97]:
# Se reemplaza la palabra km
EmpleadosAttrition['DistanceFromHome'] = (
    EmpleadosAttrition['DistanceFromHome_km']
    .str.replace('km', '')
    .astype(int)
)

6.	Borra las columnas Year, HiringDate y DistanceFromHome_km debido a que ya no son útiles.

In [98]:
EmpleadosAttrition = EmpleadosAttrition.drop(
    columns=['Year', 'HiringDate', 'DistanceFromHome_km']
)

7.	Aprovechando los ajustes que se están haciendo, la empresa desea saber si todos los departamentos tienen un ingreso promedio similar. Agrupar por departamento con sueldo

In [ ]:
SueldoPromedioDepto = EmpleadosAttrition.groupby('Department', as_index=False).agg(
    SueldoPromedio=('MonthlyIncome', 'mean')
)

SueldoPromedioDepto

,Department,SueldoPromedio
0,Human Resources,6239.888889
1,Research & Development,6804.149813
2,Sales,7188.250000


8.	La variable MonthlyIncome tiene un valor numérico muy grande comparada con las otras variables. Escala dicha variable para que tenga un valor entre 0 y 1.

In [101]:
scaler = MinMaxScaler()
EmpleadosAttrition['MonthlyIncome'] = scaler.fit_transform(
    EmpleadosAttrition[['MonthlyIncome']]
)

9.	Todo parece indicar que las variables categóricas que quedan sí son importantes para obtener la variable de salida. Convierte todas las variables categóricas que quedan a numéricas:

In [102]:
categorical_cols = [
    'BusinessTravel',
    'Department',
    'EducationField',
    'Gender',
    'JobRole',
    'MaritalStatus',
    'Attrition'
]

EmpleadosAttrition = pd.get_dummies(
    EmpleadosAttrition,
    columns=categorical_cols,
    drop_first=True
)

10.	Ahora debes hacer la evaluación de las variables para quedarte con las mejores. Calcula la correlación lineal de cada una de las variables con respecto al Attrition.

In [103]:
correlaciones = EmpleadosAttrition.corr(numeric_only=True)['Attrition_Yes'].sort_values(ascending=False)
correlaciones

,Attrition_Yes
Attrition_Yes,1.000000
MaritalStatus_Single,0.205849
JobRole_Sales Representative,0.191294
EducationField_Technical Degree,0.129104
JobRole_Laboratory Technician,0.125264
Department_Sales,0.066116
DistanceFromHome,0.052732
BusinessTravel_Travel_Rarely,0.042755
BusinessTravel_Travel_Frequently,0.035387
JobRole_Human Resources,0.032714


11.	Selecciona solo aquellas variables que tengan una correlación mayor o igual a 0.1, dejándolas en otro frame llamado EmpleadosAttritionFinal. No olvides mantener la variable de salida Attrition; esto es equivalente a borrar las que no cumplen con el límite

In [104]:
vars_importantes = correlaciones[correlaciones.abs() >= 0.1].index

In [105]:
EmpleadosAttritionFinal = EmpleadosAttrition[list(vars_importantes) + ['Attrition_Yes']]
EmpleadosAttritionFinal.head()

,Attrition_Yes,MaritalStatus_Single,JobRole_Sales Representative,EducationField_Technical Degree,JobRole_Laboratory Technician,JobRole_Research Director,EnvironmentSatisfaction,JobSatisfaction,JobInvolvement,YearsAtCompany,MonthlyIncome,YearsInCurrentRole,Age,TotalWorkingYears,JobLevel,Attrition_Yes
0,False,False,False,False,False,True,4,4,3,5.0,0.864269,4,50,32,4,False
1,False,False,False,False,False,False,2,2,3,3.0,0.207340,2,36,7,2,False
2,True,True,True,False,False,False,2,2,3,1.0,0.088062,0,21,1,1,True
3,False,True,False,False,False,False,2,2,3,8.0,0.497574,6,52,18,3,False
4,True,False,False,False,False,False,2,3,3,7.0,0.664470,6,33,15,3,True


In [ ]:
X = EmpleadosAttritionFinal.drop(columns=['Attrition_Yes'])

In [ ]:
X = X.fillna(X.mean())

12.	Crea una nueva variable llamada EmpleadosAttritionPCA formada por los componentes principales del frame EmpleadosAttritionFinal. Recuerda que el resultado del proceso PCA es un numpy array, por lo que, para hacer referencia a una columna, por ejemplo, la 0, puedes usar la instrucción EmpleadosAttritionPCA[:,0]).

In [106]:
pca = PCA()
EmpleadosAttritionPCA = pca.fit_transform(X)

In [107]:
varianza_acumulada = np.cumsum(pca.explained_variance_ratio_)

13.	Agrega el mínimo número de Componentes Principales en columnas del frame EmpleadosAttritionPCA que logren explicar el 80% de la varianza, al frame EmpleadosAttritionFinal. Puedes usar la instrucción assign, columna por columna, llamando a cada una C0, C1, etc., hasta las que vayas a agregar.

In [108]:
n_componentes = np.argmax(varianza_acumulada >= 0.80) + 1
print(n_componentes)

2


In [109]:
for i in range(n_componentes):
    EmpleadosAttritionFinal = EmpleadosAttritionFinal.assign(
        **{f'C{i}': EmpleadosAttritionPCA[:, i]}
    )

In [110]:
EmpleadosAttritionFinal.head()

,Attrition_Yes,MaritalStatus_Single,JobRole_Sales Representative,EducationField_Technical Degree,JobRole_Laboratory Technician,JobRole_Research Director,EnvironmentSatisfaction,JobSatisfaction,JobInvolvement,YearsAtCompany,MonthlyIncome,YearsInCurrentRole,Age,TotalWorkingYears,JobLevel,Attrition_Yes,C0,C1
0,False,False,False,False,False,True,4,4,3,5.0,0.864269,4,50,32,4,False,20.251856,-4.209613
1,False,False,False,False,False,False,2,2,3,3.0,0.207340,2,36,7,2,False,-6.385125,-3.696832
2,True,True,True,False,False,False,2,2,3,1.0,0.088062,0,21,1,1,True,-21.509020,1.742614
3,False,True,False,False,False,False,2,2,3,8.0,0.497574,6,52,18,3,False,13.822894,-5.840487
4,True,False,False,False,False,False,2,3,3,7.0,0.664470,6,33,15,3,True,-1.427118,4.143426


In [111]:
EmpleadosAttritionFinal.to_csv('EmpleadosAttritionFinal.csv', index=False)